In [18]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
from PIL import Image
from torchvision import transforms
from vit_pytorch.scalable_vit import ScalableViT
#from torchvision.models import resnet50
#from vit_pytorch.distill import DistillableViT, DistillWrapper
import torch.optim as optim
import torch.nn as nn
from pathlib import Path
from torchvision import datasets, models, transforms
from sklearn.model_selection import train_test_split
import shutil
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt

In [6]:
print(np.__version__)  # Should print the NumPy version
print(torch.__version__)  # Should print the PyTorch version

1.23.5
1.8.1+cu111


In [8]:
#Setup Path
FILTERED_DIR = r"D:\Adrian Leong\Coding\OneDrive_1_1-14-2025\Filtered Dataset"  # Replace with actual dataset path
AUGMENTED_DIR = os.path.join(FILTERED_DIR, "augmented")
SPLITS = ["train", "val", "test"]
RATIO = {"train": 0.8, "val": 0.1, "test": 0.1}
BATCH_SIZE = 32
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [9]:
#Ensure necessary directories
for split in SPLITS:
    for label in ["PASS", "REJECT"]:
        os.makedirs(os.path.join(AUGMENTED_DIR, split, label), exist_ok=True)

In [23]:
#Ensure necessary directories
for split in SPLITS:
    for label in ["PASS", "REJECT"]:
        os.makedirs(os.path.join(AUGMENTED_DIR, split, label), exist_ok=True)

#Data Augmentation with New Transformations
transformations = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomAffine(degrees=30, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomPerspective(distortion_scale=0.5, p=0.5),
    transforms.Resize((224, 224))
])

for label in ["PASS", "REJECT"]:
    label_dir = os.path.join(FILTERED_DIR, label)
    augmented_label_dir = os.path.join(AUGMENTED_DIR, "all", label)
    os.makedirs(augmented_label_dir, exist_ok=True)

    for image_name in os.listdir(label_dir):
        image_path = os.path.join(label_dir, image_name)
        image = Image.open(image_path).convert("RGB")

        for i in range(8):  # Augment to get 8x data
            augmented_image = transformations(image)
            augmented_image.save(os.path.join(augmented_label_dir, f"{Path(image_name).stem}_aug{i}.jpg"))

In [4]:
#Split Dataset
for label in ["PASS", "REJECT"]:
    augmented_label_dir = os.path.join(AUGMENTED_DIR,"all",label)
    images = [os.path.join(augmented_label_dir, img) for img in os.listdir(augmented_label_dir)]

    train, test_val = train_test_split(images, test_size=(1 - RATIO["train"]))
    val, test = train_test_split(test_val, test_size=(RATIO["test"] / (RATIO["val"] + RATIO["test"])))

    for split, split_images in zip(SPLITS, [train, val, test]):
        split_dir = os.path.join(AUGMENTED_DIR, split, label)
        for img_path in split_images:
            shutil.copy(img_path, split_dir)

In [19]:
#Load Data and Label It
class CustomDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []

        for label, folder in enumerate(["PASS", "REJECT"]):
            folder_path = os.path.join(root_dir, folder)
            for img_name in os.listdir(folder_path):
                self.image_paths.append(os.path.join(folder_path, img_name))
                self.labels.append(label)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

#Prepare Data Loaders
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_dataset = CustomDataset(root_dir=os.path.join(AUGMENTED_DIR, "train"), transform=transform)
val_dataset = CustomDataset(root_dir=os.path.join(AUGMENTED_DIR, "val"), transform=transform)
test_dataset = CustomDataset(root_dir=os.path.join(AUGMENTED_DIR, "test"), transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# Define Vision Transformer Model
model = ScalableViT(
    num_classes = 2,
    dim = 64,                               # starting model dimension. at every stage, dimension is doubled
    heads = (2, 4, 8, 16),                  # number of attention heads at each stage
    depth = (2, 2, 20, 2),                  # number of transformer blocks at each stage
    ssa_dim_key = (40, 40, 40, 32),         # the dimension of the attention keys (and queries) for SSA. in the paper, they represented this as a scale factor on the base dimension per key (ssa_dim_key / dim_key)
    reduction_factor = (8, 4, 2, 1),        # downsampling of the key / values in SSA. in the paper, this was represented as (reduction_factor ** -2)
    window_size = (64, 32, None, None),     # window size of the IWSA at each stage. None means no windowing needed
    dropout = 0.1,                          # attention and feedforward dropout
)




# Move model to device (GPU or CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Define Optimizer and Loss Function
#optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
#scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5, verbose=True)

# Create lists to store training and validation losses/accuracies
train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []


In [23]:
# Training Loop
num_epochs = 200
print("CUDA Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU found")
for epoch in range(num_epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Gradient clipping
        optimizer.step()
        
        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_losses.append(running_loss / len(train_loader))
    train_accuracies.append(correct / total)
    
    # Validation Phase
    model.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_losses.append(val_loss / len(val_loader))
    val_accuracies.append(correct / total)
    
    # Reduce learning rate if validation loss stops improving
    #scheduler.step(val_losses[-1])
    
    print(f"Epoch [{epoch+1}/{num_epochs}], "
          f"Train Loss: {train_losses[-1]:.4f}, Train Accuracy: {train_accuracies[-1]:.4f}, "
          f"Val Loss: {val_losses[-1]:.4f}, Val Accuracy: {val_accuracies[-1]:.4f}")

# Plot Training Metrics
def plot_metrics(train_losses, val_losses, train_accuracies, val_accuracies, num_epochs):
    epochs = range(num_epochs)
    
    plt.figure(figsize=(10, 5))
    plt.plot(epochs, train_losses, label='Training Loss')
    plt.plot(epochs, val_losses, label='Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    plt.show()
    
    plt.figure(figsize=(10, 5))
    plt.plot(epochs, train_accuracies, label='Training Accuracy')
    plt.plot(epochs, val_accuracies, label='Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.legend()
    plt.show()

plot_metrics(train_losses, val_losses, train_accuracies, val_accuracies, num_epochs)


CUDA Available: True
GPU Name: NVIDIA GeForce RTX 4070 Ti


AssertionError: height (56) or width (56) of feature map is not divisible by the window size (64, 64)

In [ ]:
# Evaluate model
model.eval()
y_true = []
y_pred = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

# Calculate Metrics
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')

print(f"Test Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")